In [ ]:
import os
import json
import numpy as np
import pandas as pd

def mark_to_numeric(mark):
    if pd.isna(mark):
        return np.nan
    mark = str(mark).strip()
    if ":" in mark:
        try:
            parts = [float(part) for part in mark.split(":")]
        except ValueError:
            return np.nan
        if len(parts) == 2:
            minutes, seconds = parts
            return minutes * 60 + seconds
        if len(parts) == 3:
            hours, minutes, seconds = parts
            return hours * 3600 + minutes * 60 + seconds
        return np.nan
    try:
        return float(mark)
    except Exception:
        return np.nan

In [ ]:
# Load and preprocess data
df = pd.read_csv('data.csv', sep=';')
# Normalize possible mark column names
mark_col = 'Mark' if 'Mark' in df.columns else ('Mark [meters or seconds]' if 'Mark [meters or seconds]' in df.columns else None)
if mark_col is not None:
    df['Performance'] = df[mark_col].apply(mark_to_numeric)
else:
    df['Performance'] = np.nan
# Dates and derived fields
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['DOB'] = pd.to_datetime(df['DOB'], errors='coerce')
df['Year'] = df['Date'].dt.year
# Event type and units
field_events = {
    'Long Jump', 'Triple Jump', 'High Jump', 'Pole Vault',
    'Shot Put', 'Discus Throw', 'Hammer Throw', 'Javelin Throw'
}
df['Event_Type'] = np.where(df['Event'].isin(field_events), 'field', 'track')
df['Unit'] = np.where(df['Event_Type'].eq('field'), 'm', 's')
df.head()

Différence performance moyenne vs record

In [ ]:
# Compute median (normal) performance and record per event/sex
normal_perf = (
    df.groupby(['Event', 'Sex', 'Event_Type', 'Unit'])['Performance']
    .median()
    .reset_index(name='normal_perf')
)
records = []
for (event, sex), group in df.groupby(['Event', 'Sex']):
    event_type = group['Event_Type'].iloc[0] if 'Event_Type' in group else 'track'
    unit = group['Unit'].iloc[0] if 'Unit' in group else ''
    median_perf = group['Performance'].median()
    if event_type == 'track':
        record = group['Performance'].min()
        gap = median_perf - record
    else:
        record = group['Performance'].max()
        gap = record - median_perf
    records.append({
        'Event': event,
        'Sex': sex,
        'record_perf': record,
        'gap': gap,
        'Unit': unit
    })
record_perf = pd.DataFrame(records)
df_diff_perf_record = normal_perf.merge(record_perf, on=['Event','Sex','Unit'])
plot_data = df_diff_perf_record.sort_values('gap', ascending=False).head(15)
data_dict = {
    'labels': (plot_data['Event'] + ' ' + plot_data['Sex']).tolist(),
    'gap': plot_data['gap'].round(2).tolist(),
    'normal_perf': plot_data['normal_perf'].round(2).tolist(),
    'record_perf': plot_data['record_perf'].round(2).tolist(),
    'unit': plot_data['Unit'].tolist()
}
os.makedirs('visualizations_data/gap_record', exist_ok=True)
with open('visualizations_data/gap_record/data.json', 'w') as f:
    json.dump(data_dict, f, indent=2)
print('gap_record written: visualizations_data/gap_record/data.json')

Évolution des records / Duels entre époques

In [ ]:
# Prepare yearly stats (mean and appropriate record per event/year/sex)
df_duel = df.dropna(subset=['Event', 'Sex', 'Performance', 'Year']).copy()
year_stats = (
    df_duel
    .groupby(['Event','Sex','Year','Event_Type','Unit'], as_index=False)
    .agg(mean=('Performance','mean'), record_min=('Performance','min'), record_max=('Performance','max'))
)
all_data = {}
for (event, sex), group in year_stats.groupby(['Event','Sex']):
    yearly_data = {}
    for _, row in group.iterrows():
        year = int(row['Year'])
        if row.get('Event_Type', 'track') == 'track':
            record = row['record_min']
        else:
            record = row['record_max']
        yearly_data[str(year)] = {
            'record': round(float(record),2) if not pd.isna(record) else None,
            'mean': round(float(row['mean']),2) if not pd.isna(row['mean']) else None,
        }
    all_data[f'{event}___{sex}'] = {
        'event': event,
        'sex': sex,
        'unit': group['Unit'].iloc[0],
        'event_type': group['Event_Type'].iloc[0],
        'years': sorted([int(y) for y in group['Year'].tolist() if not pd.isna(y)]),
        'yearly_data': yearly_data,
    }
os.makedirs('visualizations_data/duel_athletes', exist_ok=True)
with open('visualizations_data/duel_athletes/data.json', 'w') as f:
    json.dump(all_data, f, indent=2)
print('duel_athletes written: visualizations_data/duel_athletes/data.json')